# Auditoría reproducible: ponderación/balanceo y asistencia

Responde a dos observaciones del tutor Fernando Torre Mora sobre el Product Backlog:
ponderación/balanceo de las observaciones durante el entrenamiento, y disponibilidad
y posible reconstrucción de asistencia mediante la participación.

Este notebook NO reimplementa ninguna lógica: importa y ejecuta las funciones de
`RedBayesiana/codigo_red/auditoria_ponderacion_asistencia.py`. No modifica ningún
pipeline principal (LSTM ni red bayesiana), no genera predicciones nuevas, no ejecuta
la Tarjeta 7. Queda guardado con outputs para servir de respaldo reproducible al
Apéndice F (ponderación) y al Apéndice G (asistencia) de la tesis.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd()
while not (RAIZ / "RedBayesiana").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent

CODIGO_RED = RAIZ / "RedBayesiana" / "codigo_red"
sys.path.insert(0, str(CODIGO_RED))

import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

import auditoria_ponderacion_asistencia as aud

datos = aud.cargar_datos_crudos()
print(f"Filas estudiante-sesión cargadas: {len(datos)}")

Filas estudiante-sesión cargadas: 11700


## 1. Ponderación — LSTM

Verificación sobre el código real del notebook adaptado, no sobre lo recordado de él.

In [2]:
verificacion_lstm = aud.verificar_lstm_sin_ponderacion()
for k, v in verificacion_lstm.items():
    print(f"{k}: {v}")

assert verificacion_lstm["sample_weight_en_notebook_adaptado_celdas_codigo"] == 0, (
    "se esperaba que el notebook adaptado no use sample_weight"
)
assert verificacion_lstm["class_weight_en_notebook_adaptado_celdas_codigo"] == 0, (
    "se esperaba que el notebook adaptado (regresión) no use class_weight"
)
print()
print("Confirmado: el ajuste no usa sample_weight ni class_weight. Cada registro "
      "estudiante-sección pesa igual en el error cuadrático medio.")

sample_weight_en_notebook_adaptado_celdas_codigo: 0
class_weight_en_notebook_adaptado_celdas_codigo: 0
class_weight_en_notebook_original_celdas_codigo: 4

Confirmado: el ajuste no usa sample_weight ni class_weight. Cada registro estudiante-sección pesa igual en el error cuadrático medio.


## 2. Ponderación — Red bayesiana

El ajuste de CPD opera sobre estudiante-sesión. Para los nodos constantes dentro de un registro (año que cursa, sección, tamaño del grupo, objetivo), cada registro aporta tantos conteos como sesiones tenga.

In [3]:
tam = aud.tabla_tamanos(datos)
print("Tamaños por materia x trimestre x sección:")
print(tam.to_string(index=False))
print(f"\nmin={tam.n_estudiantes.min()} max={tam.n_estudiantes.max()} "
      f"media={tam.n_estudiantes.mean():.1f} desv={tam.n_estudiantes.std():.1f}")

Tamaños por materia x trimestre x sección:
                  materia trimestre  seccion  n_estudiantes
Algoritmos y Programación    2425-2        1             30
Algoritmos y Programación    2425-2        2             30
Algoritmos y Programación    2526-1        1             27
Algoritmos y Programación    2526-2        1             30
Algoritmos y Programación    2526-3        1             30
    Computación Emergente    2425-3        1             38
    Computación Emergente    2425-3        2             35
    Computación Emergente    2526-1        1             28
    Computación Emergente    2526-2        1             41
    Computación Emergente    2526-3        1             40
      Estructura de Datos    2425-3        2             30
      Estructura de Datos    2526-1        2             30
      Estructura de Datos    2526-2        2             18
      Estructura de Datos    2526-3        2             30
    Matemáticas Discretas    2526-2        1             

In [4]:
sxr = aud.sesiones_por_registro(datos)
print("Sesiones por registro estudiante-sección, por materia x trimestre:")
print(sxr.to_string())

Sesiones por registro estudiante-sección, por materia x trimestre:
                                     min  max  mean  count
materia                   trimestre                       
Algoritmos y Programación 2425-2      24   24  24.0     60
                          2526-1      24   24  24.0     27
                          2526-2      24   24  24.0     30
                          2526-3      24   24  24.0     30
Computación Emergente     2425-3      12   12  12.0     73
                          2526-1      24   24  24.0     28
                          2526-2      24   24  24.0     41
                          2526-3      24   24  24.0     40
Estructura de Datos       2425-3      24   24  24.0     30
                          2526-1      24   24  24.0     30
                          2526-2      24   24  24.0     18
                          2526-3      24   24  24.0     30
Matemáticas Discretas     2526-2      24   24  24.0     56
                          2526-3      24   24  2

### 3. Sesiones por registro/materia/trimestre — cuantificación de la asimetría

In [5]:
asimetria = aud.asimetria_ce_2425_3(sxr)
for k, v in asimetria.items():
    print(f"{k}: {v}")

assert asimetria["registros_afectados"] == 73
assert asimetria["sesiones_por_registro"] == 12
assert asimetria["peso_relativo_frente_al_resto"] == 0.5

print()
print("Confirmado: 73 registros de Computación Emergente 2425-3 aportan la mitad de "
      "conteos que el resto (12 de 24 sesiones) en los nodos año que cursa, sección, "
      "tamaño del grupo y objetivo. No se cuantifica aquí su efecto en el desempeño, "
      "y no se introduce ninguna ponderación correctora.")
print()
print("La inferencia y el cálculo de RMSE/R² (cartas 6 y 5) vuelven a tratar cada "
      "registro estudiante-sección de forma uniforme -- esa asimetría es exclusiva "
      "del ajuste de CPD, ya verificado en cartas anteriores y no reejecutado aquí "
      "para no tocar la implementación principal.")

materia: Computación Emergente
trimestre: 2425-3
registros_afectados: 73
sesiones_por_registro: 12
sesiones_por_registro_resto_del_estudio: 24
peso_relativo_frente_al_resto: 0.5

Confirmado: 73 registros de Computación Emergente 2425-3 aportan la mitad de conteos que el resto (12 de 24 sesiones) en los nodos año que cursa, sección, tamaño del grupo y objetivo. No se cuantifica aquí su efecto en el desempeño, y no se introduce ninguna ponderación correctora.

La inferencia y el cálculo de RMSE/R² (cartas 6 y 5) vuelven a tratar cada registro estudiante-sección de forma uniforme -- esa asimetría es exclusiva del ajuste de CPD, ya verificado en cartas anteriores y no reejecutado aquí para no tocar la implementación principal.


## 4. Asistencia — cobertura real y confirmación de exclusión del pipeline

In [6]:
inventario = aud.inventario_hojas_asistencia()
df_inventario = pd.DataFrame(inventario)
print("Hojas de asistencia reales encontradas en Datos Tesis Upstream/:")
df_inventario

Hojas de asistencia reales encontradas en Datos Tesis Upstream/:


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/openpyxl/reader/workbook.py:84: UserWarning: File contains an invalid specification for 0. This will be removed
  warn(msg)


,archivo,hoja,n_estudiantes,n_sesiones_con_fecha,n_celdas,celdas_llenas,pct_lleno
0,Datos Tesis Upstream/Algoritmos y Programacion...,sec 1,30,17,510,386,75.7
1,Datos Tesis Upstream/Algoritmos y Programacion...,sec 3,30,17,510,0,0.0
2,Datos Tesis Upstream/Algoritmos y Programacion...,sec 6,30,17,510,0,0.0
3,Datos Tesis Upstream/Estructura de Datos/2526-...,Asistencia,30,19,570,208,36.5


In [7]:
fuera_pipeline = aud.verificar_asistencia_fuera_del_pipeline(datos)
for k, v in fuera_pipeline.items():
    print(f"{k}: {v}")

assert fuera_pipeline["menciones_asistencia_en_generar_csv"] == 0
assert fuera_pipeline["columna_asistencia_en_datos_downstream"] is False
print()
print("Confirmado: 'asistencia' no aparece en generar_csv.py ni como columna en "
      "Datos Tesis Downstream/. Nunca entró al pipeline estandarizado.")

menciones_asistencia_en_generar_csv: 0
columna_asistencia_en_datos_downstream: False
columnas_reales_datos_downstream: ['anio_academico', 'dia_sesion', 'estudiante_id', 'materia', 'numero_lista', 'participaciones', 'seccion', 'semana', 'tema', 'tipo_sesion', 'trimestre']

Confirmado: 'asistencia' no aparece en generar_csv.py ni como columna en Datos Tesis Downstream/. Nunca entró al pipeline estandarizado.


In [8]:
historial = aud.verificar_sin_corrida_historica_asistencia()
print("Commits cuyo mensaje menciona 'asistencia':")
for c in historial["commits_con_asistencia_en_el_mensaje"]:
    print(f"  {c['commit']}  {c['mensaje']}")

print()
print("Commits que agregan o quitan la palabra 'asistencia' en algún archivo:")
for c in historial["commits_que_agregan_o_quitan_la_palabra_asistencia_en_algun_archivo"]:
    print(f"  {c['commit']}  {c['mensaje']}")

print()
print("Ninguno de estos commits corresponde a una corrida de entrenamiento o "
      "evaluación con asistencia como variable de entrada. El único commit relevante "
      "(5247881) solo agrega los archivos de datos originales, no código de modelado.")

Commits cuyo mensaje menciona 'asistencia':
  5247881  📊 Agrega datos de participaciones, asistencia y cronogramas (3 materias, 11 trimestres)

Commits que agregan o quitan la palabra 'asistencia' en algún archivo:
  1a1423f  L
  74df8f8  l
  ff42358  Y
  0111dda  Primeros csv en downstream
  ed66f45  Estandarización, falta 2 secciones de emergente 2425-3
  cdaeb8d  Tercera Versión: Routing Limpio
  7a61dcb  Segunda Versión: Pipeline Preprocesamiento, Limpieza
  9abcd84  Primera Versión: Limpiando Reusltados
  5ed72ea  J

Ninguno de estos commits corresponde a una corrida de entrenamiento o evaluación con asistencia como variable de entrada. El único commit relevante (5247881) solo agrega los archivos de datos originales, no código de modelado.


## 5. Prueba de la regla participación > 0 ⇒ presencia

In [9]:
asimetria_heuristica = aud.asimetria_participo_implica_asistio(datos)
for k, v in asimetria_heuristica.items():
    print(f"{k}: {v}")

assert asimetria_heuristica["total_filas_con_dato_de_participacion"] == 11383
assert asimetria_heuristica["participaciones_mayor_a_cero_asistencia_inferible"] == 1603
assert asimetria_heuristica["participaciones_igual_a_cero_asistencia_ambigua"] == 9780
assert asimetria_heuristica["pct_asistencia_inferible"] == 14.1
assert asimetria_heuristica["pct_asistencia_ambigua"] == 85.9

print()
print("Confirmado: la regla solo etiqueta con confianza el 14,1% de las filas "
      "(participación > 0). El 85,9% restante (participación = 0) no permite "
      "distinguir entre un estudiante ausente y uno presente que no participó.")

total_filas_con_dato_de_participacion: 11383
participaciones_mayor_a_cero_asistencia_inferible: 1603
pct_asistencia_inferible: 14.1
participaciones_igual_a_cero_asistencia_ambigua: 9780
pct_asistencia_ambigua: 85.9

Confirmado: la regla solo etiqueta con confianza el 14,1% de las filas (participación > 0). El 85,9% restante (participación = 0) no permite distinguir entre un estudiante ausente y uno presente que no participó.


## 6. Conclusión sobre la imposibilidad de una comparación válida con/sin asistencia

Sin ejecutar ninguna comparación, se deja documentada la razón cuantitativa.

In [10]:
cobertura_secciones_con_asistencia_real = df_inventario[df_inventario["pct_lleno"] > 0]
materias_con_asistencia = sorted(set(
    "Algoritmos y Programación" if "Algoritmos" in a else "Estructura de Datos"
    for a in cobertura_secciones_con_asistencia_real["archivo"]
))

print(f"Secciones-archivo con asistencia realmente poblada (pct_lleno > 0): "
      f"{len(cobertura_secciones_con_asistencia_real)} de {len(df_inventario)} pestañas revisadas")
print(f"Materias con alguna asistencia real: {materias_con_asistencia}")
print("Trimestre con asistencia real: 2526-3 (único)")
print("Materias sin ningún registro de asistencia: Computación Emergente, Matemáticas Discretas")
print()
print("CONCLUSIÓN:")
print("La asistencia real disponible se concentra en un único trimestre (2526-3) y en")
print("dos de las cuatro asignaturas. El esquema de validación cruzada usado por ambos")
print("modelos (leave-one-trimester-out) requiere más de un trimestre por asignatura.")
print("No existe ningún trimestre adicional con asistencia real para formar una segunda")
print("partición de entrenamiento y prueba. Por lo tanto, no es posible reproducir ese")
print("esquema con asistencia como variable de entrada. No se ejecuta esa comparación.")
print("No se reconstruye ni se imputa ningún valor de asistencia para forzarla.")

Secciones-archivo con asistencia realmente poblada (pct_lleno > 0): 2 de 4 pestañas revisadas
Materias con alguna asistencia real: ['Algoritmos y Programación', 'Estructura de Datos']
Trimestre con asistencia real: 2526-3 (único)
Materias sin ningún registro de asistencia: Computación Emergente, Matemáticas Discretas

CONCLUSIÓN:
La asistencia real disponible se concentra en un único trimestre (2526-3) y en
dos de las cuatro asignaturas. El esquema de validación cruzada usado por ambos
modelos (leave-one-trimester-out) requiere más de un trimestre por asignatura.
No existe ningún trimestre adicional con asistencia real para formar una segunda
partición de entrenamiento y prueba. Por lo tanto, no es posible reproducir ese
esquema con asistencia como variable de entrada. No se ejecuta esa comparación.
No se reconstruye ni se imputa ningún valor de asistencia para forzarla.


## CSV de soporte para el informe

Tablas pequeñas, derivadas directamente de los datos anteriores, para usar en los Apéndices F y G sin recalcular nada al redactar.

In [11]:
CARPETA_RESULTADOS = RAIZ / "RedBayesiana" / "resultados"
CARPETA_RESULTADOS.mkdir(parents=True, exist_ok=True)

sxr.reset_index().to_csv(CARPETA_RESULTADOS / "auditoria_sesiones_por_registro.csv", index=False)
df_inventario.to_csv(CARPETA_RESULTADOS / "auditoria_inventario_asistencia.csv", index=False)
pd.DataFrame([asimetria_heuristica]).to_csv(
    CARPETA_RESULTADOS / "auditoria_heuristica_participacion.csv", index=False
)

print("Guardados:")
for nombre in ("auditoria_sesiones_por_registro.csv", "auditoria_inventario_asistencia.csv",
               "auditoria_heuristica_participacion.csv"):
    print(f"  {CARPETA_RESULTADOS / nombre}")

Guardados:
  /Users/nelsoncarrillo/Downloads/tesis-prediccion-participacion/RedBayesiana/resultados/auditoria_sesiones_por_registro.csv
  /Users/nelsoncarrillo/Downloads/tesis-prediccion-participacion/RedBayesiana/resultados/auditoria_inventario_asistencia.csv
  /Users/nelsoncarrillo/Downloads/tesis-prediccion-participacion/RedBayesiana/resultados/auditoria_heuristica_participacion.csv


## Cierre

Ambos requerimientos del tutor quedan técnicamente sustentados con evidencia reproducible:

1. **Ponderación**: verificado por código (sin `sample_weight`/`class_weight` en el ajuste
   actual de la LSTM) y por datos (asimetría de sesiones por registro en la red bayesiana,
   cuantificada exactamente: 73 registros con la mitad de peso). No se introdujo ninguna
   ponderación correctora en ningún modelo.
2. **Asistencia**: cobertura real cuantificada, confirmada su ausencia del pipeline
   estandarizado, confirmada la ausencia de una corrida histórica, y demostrado con
   números por qué la regla "participación > 0 ⇒ presencia" no produce una variable
   utilizable ni permite una comparación leave-one-trimester-out válida.

No se modificó ningún pipeline principal, no se generaron predicciones nuevas, no se
ejecutó la Tarjeta 7, no se tocó el InformeTesis ni Trello.